# Evaluacion HPC y Cloud Computing para deteccion de ciberataques
## Notebook AZURE — Escenarios S2 (sin optimizacion) y S4 (optimizado)
**Grupo 4 — Universidad del Pacifico**

Este notebook ejecuta la **mitad cloud** del diseno factorial 2x2 sobre la VM **Standard_D4s_v3 (4 vCPU, 16 GiB) en Chile Central**. Es gemelo del notebook `HPC_CP_Grupo4_S1_S3_Local.ipynb` y usa los **mismos modulos `.py`, el mismo preprocesamiento y la misma configuracion de modelos** (con `N_CORES` identico) para que la comparacion sea valida.

**Convencion de outputs:** por escenario `S2_…`, `S4_…`; agregados del entorno `AZURE_…`.

**Requisitos en la VM:** Python 3.13 (Miniconda), `preprocessing.py`, `evaluation.py`, el dataset `UNSW-NB15_1.csv` y las librerias de `requirements.txt`. Ver el `.docx` de instrucciones para el paso a paso de despliegue (crear VM, MobaXterm, SSH, scp).

## 1. Parametros globales (identicos al notebook local)

In [ ]:
import os

# ---- Identidad de ESTE notebook ----
ENTORNO = "azure"                 # Ejecuta S2 (sin opt) y S4 (optimizado) en la VM Azure
ESCENARIOS = ["S2", "S4"]
PREFIX_ENV = "AZURE"

# ---- LIMITE DE NUCLEOS (equidad LOCAL vs AZURE) ----
# La VM Standard_D4s_v3 tiene 4 vCPU. Fijamos N_CORES=4 de forma EXPLICITA para
# que la configuracion de los modelos sea identica a la del notebook local.
N_CORES = int(os.environ.get("N_CORES", "4"))
for _v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"]:
    os.environ[_v] = str(N_CORES)

# ---- Control del experimento (IDENTICO al notebook local) ----
N_REPETICIONES = int(os.environ.get("N_REPETICIONES", "5"))
RANDOM_SEED_BASE = 42
TEST_SIZE = 0.2

# ---- Dataset (mismo nombre; en la VM colocalo junto al notebook o en ~/data) ----
DATA_FILENAME = "UNSW-NB15_1.csv"
DATA_PATH = os.environ.get("DATA_PATH", DATA_FILENAME)
FEATURES_PATH = os.environ.get("FEATURES_PATH", "NUSWNB15_features.xlsx")  # opcional

# ---- Costo de la VM Azure (CONFIRMAR el valor por HORA del Portal) ----
AZURE_VM_TYPE = "Standard_D4s_v3"     # 4 vCPU, 16 GiB
AZURE_VM_REGION = "Chile Central"
AZURE_VM_COST_PER_HOUR_USD = float(os.environ.get("AZURE_VM_COST_PER_HOUR_USD", "0.376"))
# ^ 0.376 = 274.48 USD/mes / 730 h. Reemplaza por el valor por hora exacto del Portal.

# ---- Directorios de salida ----
OUTPUT_DIR = "outputs"
FIGURES_DIR = os.path.join(OUTPUT_DIR, "figures")
TABLES_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

print(f"Entorno         : {ENTORNO.upper()}  -> escenarios {ESCENARIOS}")
print(f"VM              : {AZURE_VM_TYPE} ({AZURE_VM_REGION}) | {AZURE_VM_COST_PER_HOUR_USD} USD/h")
print(f"N_CORES         : {N_CORES}")
print(f"N_REPETICIONES  : {N_REPETICIONES}")
print(f"DATA_PATH       : {DATA_PATH}")
print(f"Outputs en      : {OUTPUT_DIR}/ (prefijos S2_, S4_, AZURE_)")

## 2. Importacion de librerias y modulos `.py`

In [ ]:
import sys, os, time, platform, socket
from datetime import datetime
import numpy as np
import pandas as pd
import psutil
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn, xgboost
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Importa los modulos .py (modularizacion solicitada por el profesor).
# Deben estar en la MISMA carpeta que este notebook.
try:
    from preprocessing import (preprocess_minimal, preprocess_optimized,
                               reduce_memory_usage, split_data, TARGET_COL)
    from evaluation import ResourceMonitor, build_models, train_evaluate_once
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Faltan preprocessing.py y/o evaluation.py en la misma carpeta que "
        "este notebook. Copialos junto al .ipynb antes de ejecutar.") from e

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
sns.set_style("whitegrid")
plt.rcParams["figure.max_open_warning"] = 0
print("Imports OK | Python", sys.version.split()[0],
      "| sklearn", sklearn.__version__, "| xgboost", xgboost.__version__)

## 3. Banner de especificaciones de la VM (pre-ejecucion)
Documenta el hardware real de la VM Azure para el paper y la reproducibilidad.

In [ ]:
def get_hardware_info():
    vm = psutil.virtual_memory()
    return {
        "entorno": ENTORNO,
        "fecha_ejecucion": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "hostname": socket.gethostname(),
        "sistema_operativo": platform.system() + " " + platform.release(),
        "arquitectura": platform.machine(),
        "python_version": sys.version.split()[0],
        "cpu_modelo": platform.processor() or "no disponible",
        "cpu_nucleos_fisicos": psutil.cpu_count(logical=False),
        "cpu_nucleos_logicos": psutil.cpu_count(logical=True),
        "n_cores_usados_experimento": N_CORES,
        "ram_total_gb": round(vm.total / (1024 ** 3), 2),
        "ram_disponible_gb": round(vm.available / (1024 ** 3), 2),
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgboost.__version__,
    }

hw = get_hardware_info()
print("=" * 64)
print(" ESPECIFICACIONES DEL SISTEMA QUE EJECUTA EL CODIGO ".center(64, "="))
print("=" * 64)
for k, v in hw.items():
    print(f"  {k:30s}: {v}")
print("=" * 64)
print(f"  NOTA: el experimento se limita a N_CORES={N_CORES} nucleos logicos")
print(f"        para una comparacion equitativa LOCAL vs AZURE.")
print("=" * 64)

hardware_df = pd.DataFrame([hw]).T.reset_index()
hardware_df.columns = ["caracteristica", "valor"]
hardware_df.to_csv(os.path.join(TABLES_DIR, f"{PREFIX_ENV}_tabla_hardware.csv"), index=False)
display(hardware_df)

## 4. Carga del dataset UNSW-NB15 (en la VM)
Mismo CSV con encabezados en la primera fila. Colocalo junto al notebook o ajusta `DATA_PATH`.

In [ ]:
def load_dataset(path):
    """Carga el CSV asumiendo que la PRIMERA FILA contiene los encabezados."""
    t0 = time.perf_counter()
    df = pd.read_csv(path, low_memory=False)
    return df, time.perf_counter() - t0

assert os.path.exists(DATA_PATH), (
    f"No se encontro el dataset en: {DATA_PATH}\n"
    f"Coloca '{DATA_FILENAME}' junto al notebook o define la variable de "
    f"entorno DATA_PATH.")

mon = ResourceMonitor(interval=0.1); mon.start()
df, load_time = load_dataset(DATA_PATH)
load_stats = mon.stop()

# Normalizacion defensiva del nombre de la columna objetivo
if "Label" not in df.columns:
    for cand in ["label", "LABEL"]:
        if cand in df.columns:
            df = df.rename(columns={cand: "Label"}); break
assert "Label" in df.columns, "No se encontro la columna objetivo 'Label'."

print(f"Dimensiones : {df.shape[0]:,} filas x {df.shape[1]} columnas")
print(f"Carga       : {load_time:.3f} s | RAM pico {load_stats['ram_peak_gb']:.2f} GB")
print(f"Memoria df  : {df.memory_usage(deep=True).sum()/(1024**2):.2f} MB")
df.head(3)

## 5. Chequeo rapido del dataset
El EDA completo (figuras) se genera en el notebook local; aqui solo se valida la carga.

In [ ]:
# Chequeo rapido del dataset (el EDA completo esta en el notebook local)
print(f"Filas    : {df.shape[0]:,} | Columnas: {df.shape[1]}")
print(f"Memoria  : {df.memory_usage(deep=True).sum()/(1024**2):.2f} MB")
print("\nBalance de clases (Label):")
print(df["Label"].value_counts(normalize=True).round(4).to_string())

## 6. Preprocesamiento modular y reduccion de memoria

In [ ]:
print(">> Preprocesamiento SIN optimizacion (S1 / S2)...")
mem_before = df.memory_usage(deep=True).sum() / (1024 ** 2)
df_no_opt, _ = preprocess_minimal(df)
mem_no_opt = df_no_opt.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"   Memoria antes  : {mem_before:.2f} MB | sin opt: {mem_no_opt:.2f} MB | shape {df_no_opt.shape}")

print(">> Preprocesamiento OPTIMIZADO (S3 / S4)...")
df_opt, _ = preprocess_optimized(df)
mem_opt = df_opt.memory_usage(deep=True).sum() / (1024 ** 2)
red = (1 - mem_opt / mem_no_opt) * 100 if mem_no_opt > 0 else 0
print(f"   Memoria optim. : {mem_opt:.2f} MB | shape {df_opt.shape} | reduccion {red:.2f}%")

memory_table = pd.DataFrame({
    "pipeline": ["Sin optimizacion", "Optimizado"],
    "memoria_mb": [round(mem_no_opt, 2), round(mem_opt, 2)],
    "reduccion_vs_sin_opt_pct": [0.0, round(red, 2)],
})
memory_table.to_csv(os.path.join(TABLES_DIR, f"{PREFIX_ENV}_tabla_optimizacion_memoria.csv"), index=False)
display(memory_table)

## 7. Funciones de ejecucion de escenarios

In [ ]:
def run_scenario(df_processed, scenario_label, scenario_type, env, n_reps, n_cores):
    """Ejecuta N repeticiones x 3 modelos para un escenario.

    Devuelve (results_df con TODAS las repeticiones, dict de matrices de
    confusion de la ultima repeticion).
    """
    results = []
    confusion_matrices = {}
    for rep in range(n_reps):
        seed = RANDOM_SEED_BASE + rep
        print(f"  [{scenario_label}] Repeticion {rep+1}/{n_reps} (seed={seed})")
        X_train, X_test, y_train, y_test = split_data(df_processed, random_state=seed)
        # Escalado SOLO para el MLP (sensible a la escala)
        scaler = StandardScaler()
        X_train_s = scaler.fit_transform(X_train)
        X_test_s = scaler.transform(X_test)
        models = build_models(scenario_type, seed, n_cores)
        for name, model in models.items():
            Xtr, Xte = (X_train_s, X_test_s) if name == "MLP" else (X_train, X_test)
            res, y_pred = train_evaluate_once(model, Xtr, Xte, y_train, y_test,
                                              name, scenario_label, rep + 1)
            res["env"] = env
            res["n_cores"] = n_cores
            results.append(res)
            print(f"    {name:13s} train={res['train_time_sec']:7.2f}s "
                  f"f1={res['f1_score']:.4f} cpu_peak={res['cpu_train_peak_pct']:5.1f}% "
                  f"conv_warn={res['convergence_warning']}")
            if rep == n_reps - 1:
                confusion_matrices[name] = confusion_matrix(y_test, y_pred)
    return pd.DataFrame(results), confusion_matrices


METRIC_COLS = ["train_time_sec", "inference_time_sec", "total_time_sec",
               "accuracy", "precision", "recall", "f1_score",
               "cpu_train_peak_pct", "cpu_train_avg_pct", "ram_train_peak_gb"]


def make_summary(results_df):
    """Resumen media +/- desviacion estandar por (escenario, modelo)."""
    s = results_df.groupby(["scenario", "model"])[METRIC_COLS].agg(["mean", "std"]).round(4)
    s.columns = ["_".join(c).strip() for c in s.columns.values]
    return s.reset_index()


def export_scenario_outputs(results_df, cms, skey):
    """Exporta detallado, resumen, confusion (tabla + PNG) con prefijo S#_."""
    results_df.to_csv(os.path.join(TABLES_DIR, f"{skey}_tabla_resultados_detallados.csv"), index=False)
    summ = make_summary(results_df)
    summ.to_csv(os.path.join(TABLES_DIR, f"{skey}_tabla_resultados_resumen.csv"), index=False)
    rows = []
    for model_name, cm in cms.items():
        tn, fp, fn, tp = cm.ravel()
        rows.append({"escenario": skey, "modelo": model_name,
                     "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
                     "tasa_FN_pct": round(fn / (fn + tp) * 100, 2) if (fn + tp) > 0 else 0})
        fig, ax = plt.subplots(figsize=(4.5, 4))
        ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=["Normal", "Ataque"]).plot(
            ax=ax, cmap="Blues", colorbar=False, values_format="d")
        ax.set_title(f"{model_name} - {skey}")
        plt.tight_layout()
        plt.savefig(os.path.join(FIGURES_DIR, f"{skey}_fig_confusion_{model_name.lower()}.png"),
                    dpi=300, bbox_inches="tight")
        plt.show(); plt.close(fig)
    pd.DataFrame(rows).to_csv(os.path.join(TABLES_DIR, f"{skey}_tabla_confusion.csv"), index=False)
    return summ

## 8. Ejecucion de escenarios cloud
### 8.1 Escenario 2 — Azure sin optimizacion

In [ ]:
print("=" * 70); print("ESCENARIO 2: AZURE SIN OPTIMIZACION"); print("=" * 70)
res_s2, cm_s2 = run_scenario(df_no_opt, "S2 - Azure sin opt", "no_opt", "azure", N_REPETICIONES, N_CORES)
sum_s2 = export_scenario_outputs(res_s2, cm_s2, "S2")
display(sum_s2[["scenario", "model", "train_time_sec_mean", "train_time_sec_std", "f1_score_mean"]])

### 8.2 Escenario 4 — Azure optimizado

In [ ]:
print("=" * 70); print("ESCENARIO 4: AZURE OPTIMIZADO"); print("=" * 70)
res_s4, cm_s4 = run_scenario(df_opt, "S4 - Azure opt", "opt", "azure", N_REPETICIONES, N_CORES)
sum_s4 = export_scenario_outputs(res_s4, cm_s4, "S4")
display(sum_s4[["scenario", "model", "train_time_sec_mean", "train_time_sec_std", "f1_score_mean"]])

## 9. Resumen consolidado Azure (S2 + S4) y costo real de la VM

In [ ]:
# Consolidado del entorno AZURE (S2 + S4) + costo real de la VM
res_azure = pd.concat([res_s2, res_s4], ignore_index=True)
res_azure.to_csv(os.path.join(TABLES_DIR, "AZURE_tabla_resultados_detallados.csv"), index=False)
summary_env = make_summary(res_azure)
summary_env.to_csv(os.path.join(TABLES_DIR, "AZURE_tabla_resultados_resumen_media_std.csv"), index=False)

OVERHEAD_SEC = 60.0
cost = res_azure.groupby(["scenario", "env"])["total_time_sec"].sum().reset_index()
cost["overhead_sec"] = OVERHEAD_SEC
cost["tiempo_total_sec"] = cost["total_time_sec"] + cost["overhead_sec"]
cost["tiempo_total_hr"] = cost["tiempo_total_sec"] / 3600
cost["costo_usd_por_hora"] = AZURE_VM_COST_PER_HOUR_USD
cost["costo_total_usd"] = cost["tiempo_total_hr"] * cost["costo_usd_por_hora"]
cost["vm_tipo"] = AZURE_VM_TYPE
cost["region"] = AZURE_VM_REGION
cost.to_csv(os.path.join(TABLES_DIR, "AZURE_tabla_costos.csv"), index=False)
display(cost.round(4))

## 10. Figuras comparativas Azure (S2 vs S4)

In [ ]:
def barplot_with_errors(df_summary, value_col, error_col, ylabel, title, save_path):
    pivot_val = df_summary.pivot(index="model", columns="scenario", values=value_col)
    pivot_err = df_summary.pivot(index="model", columns="scenario", values=error_col)
    fig, ax = plt.subplots(figsize=(8, 5))
    pivot_val.plot(kind="bar", ax=ax, yerr=pivot_err, capsize=4)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.set_xlabel("Modelo")
    plt.xticks(rotation=0); ax.legend(title="Escenario", loc="best")
    plt.tight_layout(); plt.savefig(save_path, dpi=300, bbox_inches="tight"); plt.show()

barplot_with_errors(summary_env, "train_time_sec_mean", "train_time_sec_std",
    "Tiempo de entrenamiento (s)",
    f"Tiempo de entrenamiento por modelo ({PREFIX_ENV})",
    os.path.join(FIGURES_DIR, f"{PREFIX_ENV}_fig_comparacion_tiempos.png"))

barplot_with_errors(summary_env, "f1_score_mean", "f1_score_std", "F1-score",
    f"F1-score por modelo ({PREFIX_ENV})",
    os.path.join(FIGURES_DIR, f"{PREFIX_ENV}_fig_comparacion_f1.png"))

pivot_cpu = summary_env.pivot(index="model", columns="scenario", values="cpu_train_peak_pct_mean")
fig, ax = plt.subplots(figsize=(8, 5))
pivot_cpu.plot(kind="bar", ax=ax)
ax.set_title(f"Uso pico de CPU en entrenamiento ({PREFIX_ENV})")
ax.set_ylabel("CPU pico (%)"); ax.set_xlabel("Modelo"); plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, f"{PREFIX_ENV}_fig_cpu_train.png"), dpi=300, bbox_inches="tight")
plt.show()

## 11. Libro Excel del entorno Azure

In [ ]:
# Libro Excel del entorno AZURE
xlsx_path = os.path.join(OUTPUT_DIR, "AZURE_resultados.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    hardware_df.to_excel(xw, sheet_name="hardware", index=False)
    memory_table.to_excel(xw, sheet_name="memoria", index=False)
    res_azure.to_excel(xw, sheet_name="detallado_S2_S4", index=False)
    summary_env.to_excel(xw, sheet_name="resumen_media_std", index=False)
    pd.read_csv(os.path.join(TABLES_DIR, "S2_tabla_confusion.csv")).to_excel(xw, sheet_name="confusion_S2", index=False)
    pd.read_csv(os.path.join(TABLES_DIR, "S4_tabla_confusion.csv")).to_excel(xw, sheet_name="confusion_S4", index=False)
    cost.to_excel(xw, sheet_name="costos_azure", index=False)
print(f"Libro Excel generado: {xlsx_path}")

## 12. Cierre y siguiente paso

In [ ]:
print("=" * 64)
print(" PASO SIGUIENTE: TRAER LOS RESULTADOS A LOCAL ".center(64, "="))
print("=" * 64)
print("Desde tu PC local, descarga toda la carpeta outputs/ de la VM, por ej.:")
print("  scp -r -i <llave.pem> azureuser@<IP_VM>:~/proyecto/outputs/ .")
print("Asegurate de copiar los AZURE_*.csv a outputs/tables/ junto a los LOCAL_*.")
print("Luego, en el notebook LOCAL, re-ejecuta la Seccion 13 (consolidacion final).")
print("\nNO OLVIDES APAGAR LA VM para no gastar credito:")
print("  az vm deallocate -g <grupo_recursos> -n <nombre_vm>")